<a href="https://colab.research.google.com/github/tixomirof/mo2_team_colabs/blob/main/%D0%9C%D0%9E2_%D0%9A%D0%BE%D0%BC%D0%B0%D0%BD%D0%B4%D0%B02_%D0%9B%D0%B0%D0%B102_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Кросс-валидация

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split, cross_validate, cross_val_score

Загрузим датасет, полученный в результате первой лабораторной работы

In [2]:
df = pd.read_csv("Engineered_Data.csv")
df.head()

,Employment,YearsCode,YearsCodePro,PreviousSalary,ComputerSkills,Employed,APL,ASP.NET,ASP.NET Core,AWS,...,Age_>35,Accessibility_Yes,EdLevel_NoHigherEd,EdLevel_Other,EdLevel_PhD,EdLevel_Undergraduate,Gender_NonBinary,Gender_Woman,MentalHealth_Yes,MainBranch_NotDev
0,1,7.0,4,51552.0,4,0,0,0,0,0,...,False,False,False,False,False,False,False,False,False,False
1,1,12.0,5,46482.0,12,1,0,0,0,1,...,False,False,False,False,False,True,False,False,False,False
2,1,15.0,6,77290.0,7,0,0,0,0,0,...,False,False,False,False,False,False,False,False,False,False
3,1,9.0,6,46135.0,13,0,0,0,0,1,...,False,False,False,False,False,True,False,False,False,False
4,0,40.0,30,160932.0,2,0,0,0,0,0,...,True,False,False,False,True,False,False,False,False,True


Разделим данные, выделим признаки X и целевую переменную y

In [3]:
X = df.drop('Employed',axis=1)
y = df['Employed']

##Обучающий и тестовый наборы данных

Разделим данные на обучающий (70%) и тестовый (30%) наборы данных

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=101)

Масштабируем данные

In [5]:
scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

Создадим и обучим модель на обучающих данных

In [6]:
model = Ridge(alpha=1)
model.fit(X_train,y_train)

Ridge(alpha=1)

Оценим модель на тестовых данных

In [7]:
y_pred = model.predict(X_test)

In [8]:
mean_squared_error(y_test,y_pred)

0.08341244261908598

##Обучающий, валидационный и тестовый наборы данных

Разделим данные на обучающий (70%), валидационный (15%) и тестовый (15%) наборы данных

In [9]:
X_train, X_OTHER, y_train, y_OTHER = train_test_split(X, y, test_size=0.3, random_state=101)

X_eval, X_test, y_eval, y_test = train_test_split(X_OTHER, y_OTHER, test_size=0.5, random_state=101)

Масштабируем данные

In [10]:
scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_eval = scaler.transform(X_eval)
X_test = scaler.transform(X_test)

Создадим и обучим модель на обучающих данных

In [11]:
model = Ridge(alpha=1)
model.fit(X_train,y_train)

Ridge(alpha=1)

Оценим модель на валидационных данных

In [12]:
y_eval_pred = model.predict(X_eval)

In [13]:
mean_squared_error(y_eval,y_eval_pred)

0.08340350200894102

Вычислим финальные метрики на тестовом наборе данных

In [14]:
y_final_test_pred = model.predict(X_test)
mean_squared_error(y_test,y_final_test_pred)

0.08342138322923091

##Кросс-валидация с помощью cross_val_score

Разделим данные на обучающий (70%) и тестовый (30%) наборы данных

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=101)

Масштабируем данные

In [16]:
scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

Создадим модель

In [17]:
model = Ridge(alpha=1)

In [18]:
scores = cross_val_score(model,X_train,y_train,
                         scoring='neg_mean_squared_error',cv=5)

In [19]:
scores

array([-0.08350631, -0.08324359, -0.08266161, -0.08440776, -0.08525137])

Выведем среднее значение MSE scores

In [20]:
abs(scores.mean())

np.float64(0.08381412766211578)

Обучим модель

In [21]:
model.fit(X_train,y_train)

Ridge(alpha=1)

Вычислим финальные метрики на тестовом наборе данных

In [22]:
y_final_test_pred = model.predict(X_test)

In [23]:
mean_squared_error(y_test,y_final_test_pred)

0.08341244261908598

##Кросс-валидация с помощью cross_validate

Разделим данные на обучающий (70%) и тестовый (30%) наборы данных

In [24]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=101)

Масштабируем данные

In [25]:
scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

Создадим модель

In [26]:
model = Ridge(alpha=1)

In [27]:
scores = cross_validate(model,X_train,y_train,
                         scoring=['neg_mean_absolute_error','neg_mean_squared_error','max_error'],cv=5)

In [28]:
pd.DataFrame(scores)

,fit_time,score_time,test_neg_mean_absolute_error,test_neg_mean_squared_error,test_max_error
0,0.140496,0.007858,-0.246657,-0.083506,-0.967970
1,0.162924,0.022564,-0.245941,-0.083244,-1.007668
2,0.203300,0.013611,-0.245686,-0.082662,-0.946453
3,0.189605,0.007292,-0.247270,-0.084408,-1.030191
4,0.154006,0.005781,-0.250162,-0.085251,-0.978276


In [29]:
pd.DataFrame(scores).mean()

,0
fit_time,0.170066
score_time,0.011421
test_neg_mean_absolute_error,-0.247143
test_neg_mean_squared_error,-0.083814
test_max_error,-0.986112


Обучим модель

In [30]:
model.fit(X_train,y_train)

Ridge(alpha=1)

Вычислим финальные метрики на тестовом наборе данных

In [31]:
y_final_test_pred = model.predict(X_test)

In [32]:
mean_squared_error(y_test,y_final_test_pred)

0.08341244261908598

#Поиск по сетке

До этого мы использовали только Ridge ($L_2$) регуляризацию. Сейчас с помощью поиска по сетке (GridSearchCV) определим наилучший способ регуляризации и наилучшие параметры.

Будем оценивать следующие модели регуляризации:
*   $L_1$-регуляризация (Lasso);
*   $L_2$-регуляризация (Ridge);
*   $L_1+L_2$ регуляризация (ElasticNET).

Для $L_1$-регуляризации используем следующие параметры:
*   `Alpha` - коэффициент регулязирации;
*   `Solver` - алгоритм оптимизации:
    * 'auto' - выбирается автоматически;
    * 'svd' - сингулярное разложение (для маленьких данных);
    * 'cholesky' - для плотных матриц;
    * 'lsqr' - итеративный метод (для больших данных);
    * 'sag' - стохастический градиент (для очень больших данных).

Для $L_2$-регуляризации:
*   `Alpha` - коэффициент регуляризации;
*   `MaxIter` - максимальное количество итераций для сходимости алгоритма;
*   `Tol` - допуск сходимости (остановка, когда улучшения меньше этого значения).

Для $L_1+L_2$ регуляризации:
*   `Alpha` - коэффициент регуляризации;
*   `L1Ratio` - баланс между $L_1$ и $L_2$ регуляризацией;
*   `MaxIter` - максимальное количество итераций для сходимости алгоритма;
*   `Tol` - допуск сходимости (остановка, когда улучшения меньше этого значения).

В целом можно обойтись без тестирования моделей $L_1$ и $L_2$ , просто указав значения 0 и 1 для коэффициента `l1_ratio`, но в случае с $L_1$-регуляризацией нельзя будет указать `solver`. Чтобы поэкспериментировать, напишем алгоритм, который проверит каждую модель.

In [33]:
# 1. Ridge регрессия
ridge_params = {
    'alpha': [0.001, 0.01, 0.1, 1, 10, 100],
    'solver': ['auto', 'svd', 'cholesky', 'lsqr', 'sag']
}

ridge = Ridge()
ridge_grid = GridSearchCV(
    ridge,
    ridge_params,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1
)
ridge_grid.fit(X_train, y_train)

# 2. Lasso регрессия
lasso_params = {
    'alpha': [0.001, 0.01, 0.1, 1, 10, 100],
    'max_iter': [1000, 5000, 10000],
    'tol': [0.0001, 0.001]
}

lasso = Lasso()
lasso_grid = GridSearchCV(
    lasso,
    lasso_params,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1
)
lasso_grid.fit(X_train, y_train)

# 3. ElasticNet регрессия
elastic_params = {
    'alpha': [0.001, 0.01, 0.1, 1, 10],
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9, 1],
    'max_iter': [1000, 5000],
    'tol': [0.0001, 0.001]
}

elastic = ElasticNet()
elastic_grid = GridSearchCV(
    elastic,
    elastic_params,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1
)
elastic_grid.fit(X_train, y_train)

# Сравнение результатов
models = {
    'Ridge': ridge_grid,
    'Lasso': lasso_grid,
    'ElasticNet': elastic_grid
}

results = []
for name, model in models.items():
    results.append({
        'Model': name,
        'Best Params': model.best_params_,
        'Best CV Score': -model.best_score_,
        'Test MSE': mean_squared_error(y_test, model.predict(X_test)),
        'Test R2': r2_score(y_test, model.predict(X_test))
    })

results_df = pd.DataFrame(results)
print("Сравнение моделей:")
print(results_df)

# Выбор лучшей модели
best_model_name = results_df.loc[results_df['Test MSE'].idxmin(), 'Model']
best_model = models[best_model_name].best_estimator_
print(f"\nЛучшая модель: {best_model_name}")
print(f"Параметры: {models[best_model_name].best_params_}")

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Fitting 5 folds for each of 36 candidates, totalling 180 fits
Fitting 5 folds for each of 120 candidates, totalling 600 fits
Сравнение моделей:
        Model                                        Best Params  \
0       Ridge                    {'alpha': 100, 'solver': 'sag'}   
1       Lasso  {'alpha': 0.001, 'max_iter': 1000, 'tol': 0.0001}   
2  ElasticNet  {'alpha': 0.001, 'l1_ratio': 0.5, 'max_iter': ...   

   Best CV Score  Test MSE   Test R2  
0       0.083813  0.083415  0.664552  
1       0.083792  0.083436  0.664467  
2       0.083779  0.083398  0.664621  

Лучшая модель: ElasticNet
Параметры: {'alpha': 0.001, 'l1_ratio': 0.5, 'max_iter': 1000, 'tol': 0.001}


Как итог, наилучшим объектом регуляризации стала модель ElasticNet (регуляризация $L_1+L_2$), со следующими параметрами:

*   `alpha` (коэффициент регуляризации) $= 0.001$;
*   `l1_ratio` (коэффициент использования $L_1$-регуляризации) $= 0.5$ ($50\%$);
*   `max_iter` (максимальное количество итераций) $= 1000$;
*   `tol` (допуск сходимости) $= 0.001$.


Напишем `Pipeline` для воспроизведения модели регуляризации в будущем.


In [35]:
elasticnet_pipeline = Pipeline([
    ('scaler', StandardScaler()),  # Масштабирование признаков
    ('elasticnet', ElasticNet(
        alpha=0.001,
        l1_ratio=0.5,
        max_iter=1000,
        tol=0.001,
        random_state=42  # для воспроизводимости
    ))
])

joblib.dump(elasticnet_pipeline, 'resDataElasticNetPipeline.pkl')

['resDataElasticNetPipeline.pkl']

Ниже приведен код загрузки Pipeline для использования его в будущих лабораторных работах.

In [ ]:
loaded_pipeline = joblib.load('resDataElasticNetPipeline.pkl')